# 🎯 Watermark Detection YOLO Training - Google Colab

## Smart Auto-Labeling Strategy:
✅ **Use your manual coordinates** to automatically label all frames!
✅ **No manual annotation needed** - we expand your known regions slightly
✅ **Accurate labels** - based on where you know watermarks actually appear

## Workflow:
1. Upload your videos
2. Extract frames  
3. Auto-label using your manual coordinates (with slight expansion)
4. Train YOLO model
5. Download best.pt → Use in your code

⏰ **Total time: 20-30 minutes** (fully automated!)

---

## Step 1: Install YOLO

In [ ]:
!pip install ultralytics roboflow supervision -q
print("✅ Installation complete!")

## Step 2: Upload Your Videos

Click the folder icon on the left → Upload:
- Your videos (.mp4 files with embedded videos + black borders)

**Note:** The notebook will automatically detect and crop to the embedded video area, just like your `video_editing.py` does!

In [ ]:
import os
import cv2
import numpy as np
from google.colab import files
import shutil

# Create folders
!mkdir -p dataset/images dataset/labels dataset/videos

print("📁 Folders created")
print("\n👉 Upload your videos using the folder icon on the left")
print("   - Put videos in: dataset/videos/")
print("\n💡 The notebook will auto-detect and crop embedded videos")
print("   (same logic as your video_editing.py)")

## Step 3: Extract & Crop Frames

This will:
1. Extract every 30th frame from videos
2. Detect embedded video area (black border detection)
3. Crop to just the embedded video
4. Save cropped frames (same as your video_editing.py output)

In [ ]:
# Extract frames from videos AND crop to embedded video area
# (Same as your video_editing.py crop logic)

video_folder = 'dataset/videos'
output_folder = 'dataset/images'
frame_interval = 5  # Extract every 30th frame

def detect_video_area(frame):
    """
    Detects the embedded video area by scanning from center outward.
    Same logic as your video_editing.py
    """
    height, width = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    black_thresh = 10
    black_ratio_thresh = 0.95

    center_x = width // 2
    center_y = height // 2

    # Find top boundary
    for y in range(center_y, 0, -1):
        row = gray[y, :]
        if np.mean(row < black_thresh) > black_ratio_thresh:
            start_y = y + 1
            break
    else:
        start_y = 0

    # Find bottom boundary
    for y in range(center_y, height):
        row = gray[y, :]
        if np.mean(row < black_thresh) > black_ratio_thresh:
            end_y = y - 1
            break
    else:
        end_y = height - 1

    # Find left boundary
    for x in range(center_x, 0, -1):
        col = gray[:, x]
        if np.mean(col < black_thresh) > black_ratio_thresh:
            start_x = x + 1
            break
    else:
        start_x = 0

    # Find right boundary
    for x in range(center_x, width):
        col = gray[:, x]
        if np.mean(col < black_thresh) > black_ratio_thresh:
            end_x = x - 1
            break
    else:
        end_x = width - 1

    box_width = end_x - start_x
    box_height = end_y - start_y

    if box_width <= 0 or box_height <= 0:
        return None

    crop_margin = 5
    return (start_x + crop_margin, start_y + crop_margin,
            box_width - crop_margin, box_height - crop_margin)

print("🎬 Extracting and cropping frames...")
frame_count = 0

for video_file in os.listdir(video_folder):
    if not video_file.endswith('.mp4'):
        continue

    video_path = os.path.join(video_folder, video_file)
    video_name = os.path.splitext(video_file)[0]

    print(f"Processing: {video_file}")

    cap = cv2.VideoCapture(video_path)

    # Detect video area from first frame
    ret, first_frame = cap.read()
    if not ret:
        print(f"  ❌ Could not read {video_file}")
        continue

    video_area = detect_video_area(first_frame)
    if not video_area:
        print(f"  ❌ Could not detect video area in {video_file}")
        continue

    x, y, w, h = video_area
    print(f"  📐 Detected area: {w}x{h} at ({x}, {y})")

    # Reset to beginning
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    count = 0
    extracted = 0

    while cap.isOpened() and extracted < 500:  # Max 50 frames per video
        ret, frame = cap.read()
        if not ret:
            break

        if count % frame_interval == 0:
            # CROP to embedded video area (like your video_editing.py)
            cropped_frame = frame[y:y+h, x:x+w]

            output_path = os.path.join(output_folder, f"{video_name}_frame_{count:05d}.jpg")
            cv2.imwrite(output_path, cropped_frame)
            extracted += 1
            frame_count += 1

        count += 1

    cap.release()
    print(f"  ✅ Extracted and cropped {extracted} frames")

print(f"\n✅ Total extracted: {frame_count} CROPPED frames")
print("📍 These frames match your video_editing.py cropped output")

## Step 4: Auto-Label Using Your Manual Coordinates

We'll use your known watermark regions and expand them slightly for better coverage!

In [ ]:
# Your manual watermark regions from video_editing.py
# These are normalized coordinates (0.0 to 1.0)
manual_regions = [
    {'x': 0.5, 'y': 1.0, 'width': 0.15, 'height': 0.70},  # Center-down watermark
]

# Expansion factor to make sure we cover the watermark completely
expansion_factor = 1.1  # 10% larger in each direction

print(f"📍 Using {len(manual_regions)} known watermark regions")
print(f"📏 Expanding regions by {(expansion_factor-1)*100:.0f}% for better coverage\n")

# Auto-label all frames
images_folder = 'dataset/images'
labels_folder = 'dataset/labels'
os.makedirs(labels_folder, exist_ok=True)

labeled_count = 0
total_labels = 0

for img_file in os.listdir(images_folder):
    if not img_file.endswith('.jpg'):
        continue

    img_path = os.path.join(images_folder, img_file)
    frame = cv2.imread(img_path)
    frame_h, frame_w = frame.shape[:2]

    # Create label file
    label_file = os.path.join(labels_folder, img_file.replace('.jpg', '.txt'))

    with open(label_file, 'w') as f:
        for region in manual_regions:
            # Get original region
            x_norm = region['x']
            y_norm = region['y']
            w_norm = region['width']
            h_norm = region['height']

            # Expand the region
            w_expanded = w_norm * expansion_factor
            h_expanded = h_norm * expansion_factor

            # Keep center the same, just expand size
            # (YOLO format uses center_x, center_y, width, height)

            # Write label: class_id center_x center_y width height (all normalized)
            f.write(f"0 {x_norm:.6f} {y_norm:.6f} {w_expanded:.6f} {h_expanded:.6f}\n")
            total_labels += 1

    labeled_count += 1

    if labeled_count % 50 == 0:
        print(f"Labeled {labeled_count} images...")

print(f"\n✅ Auto-labeled {labeled_count} images with {total_labels} watermark regions!")
print(f"   Average: {total_labels/labeled_count:.1f} watermarks per frame")

## Step 4.1: Visualize Labeled Data 👁️

Let's preview some labeled frames to verify the labels look correct before training!

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

def draw_yolo_boxes(image_path, label_path):
    """Draw YOLO format bounding boxes on image"""
    # Read image
    img = cv2.imread(image_path)
    img_h, img_w = img.shape[:2]
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Read labels
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            lines = f.readlines()

        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id, x_center, y_center, width, height = map(float, parts)

                # Convert from YOLO format (normalized) to pixel coordinates
                x_center_px = int(x_center * img_w)
                y_center_px = int(y_center * img_h)
                box_w = int(width * img_w)
                box_h = int(height * img_h)

                # Calculate top-left corner
                x1 = int(x_center_px - box_w / 2)
                y1 = int(y_center_px - box_h / 2)
                x2 = int(x_center_px + box_w / 2)
                y2 = int(y_center_px + box_h / 2)

                # Draw rectangle (green color, thickness 2)
                cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

                # Add label text
                label_text = f"Watermark {int(class_id)}"
                cv2.putText(img_rgb, label_text, (x1, y1-10),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    return img_rgb

# ========================================
# 🎨 CUSTOMIZE HOW MANY SAMPLES TO VIEW
# ========================================
# Change this number to see more/fewer samples:
# - 6 samples = 2x3 grid (quick preview)
# - 12 samples = 3x4 grid (good overview)
# - 20 samples = 4x5 grid (thorough check)
# - 'all' = see ALL labeled images (use for small datasets)

SAMPLES_TO_SHOW = 12  # Change this! Options: 6, 12, 20, or 'all'

# Visualize samples
images_folder = 'dataset/images'
labels_folder = 'dataset/labels'

# Get all images
all_imgs = [f for f in os.listdir(images_folder) if f.endswith('.jpg')]

# Determine number of samples
if SAMPLES_TO_SHOW == 'all':
    num_samples = len(all_imgs)
    random_samples = all_imgs
else:
    num_samples = min(SAMPLES_TO_SHOW, len(all_imgs))
    random_samples = np.random.choice(all_imgs, num_samples, replace=False)

# Calculate grid size
if num_samples <= 6:
    rows, cols = 2, 3
elif num_samples <= 12:
    rows, cols = 3, 4
elif num_samples <= 20:
    rows, cols = 4, 5
else:
    # For many images, make a square-ish grid
    cols = int(np.ceil(np.sqrt(num_samples)))
    rows = int(np.ceil(num_samples / cols))

print(f"📊 Showing {num_samples} labeled samples in {rows}x{cols} grid:\n")

fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3.5))
if num_samples == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, img_file in enumerate(random_samples):
    img_path = os.path.join(images_folder, img_file)
    label_path = os.path.join(labels_folder, img_file.replace('.jpg', '.txt'))

    # Draw boxes
    img_with_boxes = draw_yolo_boxes(img_path, label_path)

    # Display
    axes[idx].imshow(img_with_boxes)
    axes[idx].set_title(f"{img_file}", fontsize=8)
    axes[idx].axis('off')

# Hide extra subplots if grid is larger than samples
for idx in range(num_samples, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✅ Showing {num_samples}/{len(all_imgs)} total frames")
print("💡 Green boxes show where watermarks will be detected")
print("🔍 Check if boxes cover the watermark regions correctly")
print("\n💡 TIP: Change SAMPLES_TO_SHOW above to see more/fewer images")
print("   - 6 = quick preview")
print("   - 12 = good overview (default)")
print("   - 20 = thorough check")
print("   - 'all' = every single frame")
print("\nIf boxes look good, continue to Step 5. If not, adjust manual_regions in Step 4.")

## Step 5: Split Dataset (Train/Val)

In [ ]:
import random

# Create train/val folders
!mkdir -p dataset/images/train dataset/images/val
!mkdir -p dataset/labels/train dataset/labels/val

# Get all images
all_images = [f for f in os.listdir('dataset/images') if f.endswith('.jpg')]
random.shuffle(all_images)

# Split 80% train, 20% val
split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

# Move files
for img in train_images:
    shutil.move(f'dataset/images/{img}', f'dataset/images/train/{img}')
    label = img.replace('.jpg', '.txt')
    if os.path.exists(f'dataset/labels/{label}'):
        shutil.move(f'dataset/labels/{label}', f'dataset/labels/train/{label}')

for img in val_images:
    shutil.move(f'dataset/images/{img}', f'dataset/images/val/{img}')
    label = img.replace('.jpg', '.txt')
    if os.path.exists(f'dataset/labels/{label}'):
        shutil.move(f'dataset/labels/{label}', f'dataset/labels/val/{label}')

print(f"✅ Dataset split:")
print(f"   Train: {len(train_images)} images")
print(f"   Val: {len(val_images)} images")

## Step 6: Create data.yaml

In [ ]:
# Create YOLO dataset config
data_yaml = """
path: /content/dataset
train: images/train
val: images/val

names:
  0: watermark
"""

with open('dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ data.yaml created")

## Step 7: Train YOLO Model 🚀

In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO('yolov8x.pt')  # Nano = fast

# Train
print("🏋️ Starting training...")
print("This will take 10-20 minutes on Colab GPU\n")

results = model.train(
    data='dataset/data.yaml',
    epochs=200,  # Fewer epochs = faster
    imgsz=640,
    batch=16,
    name='watermark_detector',
    device=0,  # Use GPU
    patience=30
)

print("\n✅ Training complete!")
print("Model saved to: runs/detect/watermark_detector/weights/best.pt")

## Step 8: Test Model

In [ ]:
from IPython.display import Image as IPImage, display

# Load trained model
model = YOLO('runs/detect/watermark_detector/weights/best.pt')

# Test on a validation image
test_images = os.listdir('dataset/images/val')
if test_images:
    test_img = f'dataset/images/val/{test_images[2]}'
    results = model(test_img, conf=0.05)

    # Save result
    annotated = results[0].plot()
    cv2.imwrite('test_result.jpg', annotated)

    print("✅ Test result:")
    display(IPImage('test_result.jpg'))

    # Print detections
    for box in results[0].boxes:
        conf = box.conf[0].cpu().numpy()
        print(f"Watermark detected - confidence: {conf:.2f}")
else:
    print("No validation images found")

## Step 9: Download Trained Model ⬇️

**This is the file you need for your local code!**

In [ ]:
from google.colab import files

# Download the trained model
model_path = 'runs/detect/watermark_detector/weights/best.pt'

print("📥 Downloading trained model...")
files.download(model_path)

print("\n✅ Download complete!")
print("\n📋 Next Steps:")
print("1. Save downloaded 'best.pt' to: VIDEO_EDITING_WATERMARK/models/")
print("2. In your local code, the model will auto-detect watermarks!")
print("3. No more manual coordinates needed!")

---

# 🎉 Done!

## What you got:
- ✅ Trained YOLO model (best.pt)
- ✅ Auto-detects watermarks
- ✅ Works with any video

## Use it in your code:
Just put `best.pt` in `VIDEO_EDITING_WATERMARK/models/` and run your video_editing.py!